<a href="https://colab.research.google.com/github/shivamkumar359/03-rd-SEM--ASSIGNMENTS/blob/main/IDB/W10/Q2/ASSIGNMENT_W10_Q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import csv
import zipfile
from pathlib import Path
from dataclasses import dataclass, field


# Database initial state
database = {
    "A": 100,
    "B": 200,
    "C": 300,
    "D": 400
}

# Global timestamp counter
timestamp_counter = 0

# Transaction execution history
transaction_log = []

# Successfully committed transactions
committed_transactions = []

# Failed validation details
failure_details = []


# ==================================================
# TRANSACTION CLASS
# ==================================================

@dataclass
class Transaction:

    transaction_id: str

    read_set: set = field(default_factory=set)

    write_set: dict = field(default_factory=dict)

    start_ts: int = 0

    validation_ts: int = 0

    finish_ts: int = 0

    status: str = "ACTIVE"


# ==================================================
# TIMESTAMP GENERATOR
# ==================================================

def generate_timestamp():

    global timestamp_counter

    timestamp_counter += 1

    return timestamp_counter

In [10]:
# ==================================================
# PHASE 1: READ / EXECUTION
# ==================================================

def read_execution(transaction, key):

    print(
        f"{transaction.transaction_id}: "
        f"Reading {key}"
    )

    if key not in database:

        print("ERROR: Data item does not exist.")

        return None

    value = database[key]

    transaction.read_set.add(key)

    print(
        f"Read successful: {key} = {value}"
    )

    return value


def write_execution(transaction, key, value):

    print(
        f"{transaction.transaction_id}: "
        f"Preparing write {key} = {value}"
    )

    transaction.write_set[key] = value


In [11]:
# ==================================================
# PHASE 2: VALIDATION
# ==================================================

def validate_transaction(transaction):

    print(
        f"\nValidating {transaction.transaction_id}"
    )

    # Assign Validation Timestamp
    transaction.validation_ts = generate_timestamp()

    print(
        f"Start-TS: {transaction.start_ts}"
    )

    print(
        f"Validation-TS: {transaction.validation_ts}"
    )

    # Compare with previously committed transactions
    for previous in committed_transactions:

        # Only transactions validated after this
        # transaction started can conflict.

        if previous.validation_ts <= transaction.start_ts:
            continue

        # Check read-set intersection with previous write-set
        conflict = (
            transaction.read_set &
            set(previous.write_set.keys())
        )

        if conflict:

            transaction.status = "ABORTED"

            transaction.finish_ts = generate_timestamp()

            message = (
                f"Transaction {transaction.transaction_id} "
                f"failed validation.\n"
                f"Conflicting previous transaction: "
                f"{previous.transaction_id}\n"
                f"Current transaction read-set: "
                f"{sorted(transaction.read_set)}\n"
                f"Previous transaction write-set: "
                f"{sorted(previous.write_set.keys())}\n"
                f"Intersecting data items: "
                f"{sorted(conflict)}\n"
                f"Reason: The transaction read data that was "
                f"modified by a transaction that belongs "
                f"to its validation interval.\n"
                f"Decision: ABORTED - Changes not committed.\n"
            )

            failure_details.append(message)

            print("VALIDATION FAILED!")

            print(
                "Conflicting items:",
                conflict
            )

            return False

    print("VALIDATION SUCCESSFUL!")

    return True

In [12]:
# ==================================================
# PHASE 3: WRITE / COMMIT
# ==================================================

def write_phase(transaction):

    print(
        f"\nWrite phase: {transaction.transaction_id}"
    )

    # Commit changes only after successful validation
    for key, value in transaction.write_set.items():

        database[key] = value

        print(
            f"Database updated: {key} = {value}"
        )

    transaction.finish_ts = generate_timestamp()

    transaction.status = "COMMITTED"

    committed_transactions.append(transaction)

    print(
        f"Transaction {transaction.transaction_id} "
        f"COMMITTED."
    )

    print(
        f"Finish-TS: {transaction.finish_ts}"
    )


In [13]:
# ==================================================
# COMPLETE TRANSACTION PROCESS
# ==================================================

def execute_transaction(transaction, read_operations, write_operations):

    print("\n" + "=" * 60)

    print(
        f"TRANSACTION: {transaction.transaction_id}"
    )

    print("=" * 60)

    # Start timestamp
    transaction.start_ts = generate_timestamp()

    print(
        f"Start-TS: {transaction.start_ts}"
    )

    # PHASE 1
    print("\nPHASE 1: READ / EXECUTION")

    for key in read_operations:

        read_execution(transaction, key)

    for key, value in write_operations.items():

        write_execution(transaction, key, value)

    # PHASE 2
    print("\nPHASE 2: VALIDATION")

    valid = validate_transaction(transaction)

    # PHASE 3
    if valid:

        print("\nPHASE 3: WRITE")

        write_phase(transaction)

    else:

        print(
            "\nPHASE 3: WRITE SKIPPED "
            "(Transaction aborted)"
        )

    # Record transaction timestamps
    transaction_log.append({

        "Transaction_ID": transaction.transaction_id,

        "Start_TS": transaction.start_ts,

        "Validation_TS": transaction.validation_ts,

        "Finish_TS": transaction.finish_ts,

        "Status": transaction.status

    })


In [14]:
# ==================================================
# CSV FILE GENERATION
# ==================================================

def generate_csv():

    filename = "transaction_log.csv"

    with open(
        filename,
        "w",
        newline=""
    ) as file:

        writer = csv.DictWriter(

            file,

            fieldnames=[
                "Transaction_ID",
                "Start_TS",
                "Validation_TS",
                "Finish_TS",
                "Status"
            ]

        )

        writer.writeheader()

        writer.writerows(transaction_log)

    print(
        f"\nCSV generated: {filename}"
    )

In [15]:
# ==================================================
# TROUBLESHOOTING REPORT
# ==================================================

def generate_failure_report():

    filename = "validation_failure.txt"

    report = """

TASK 2: VALIDATION FAILURE ANALYSIS
===================================

1. INTRODUCTION

Validation-based concurrency control executes transactions
in three phases: Read/Execution, Validation, and Write.

Each transaction receives three timestamps:
Start-TS, Validation-TS, and Finish-TS.

The Validation-TS determines the serializability order.

2. VALIDATION FAILURE SCENARIO

Consider two transactions T1 and T2.

Transaction T1 reads data item A and prepares to write B.

Transaction T2 reads data item B and prepares to write C.

T1 starts before T2 but T2 may complete validation and commit
before T1 reaches its validation phase.

When T1 is validated, the system checks its read-set against
the write-set of previously validated transactions.

If T2 has modified B, then:

T1 Read-Set = {A, B}
T2 Write-Set = {B, C}

Intersection = {B}

Since the intersection is not empty, T1 fails validation.

3. REASON FOR FAILURE

T1 previously read B, but T2 modified B before T1 was validated.

Allowing T1 to commit could violate the required serializable
execution order.

4. SOLUTION

The validation phase detects the conflict by comparing
read-sets and write-sets.

The conflicting transaction is aborted.

Its prepared changes are not applied to the database.

5. CONCLUSION

Validation-based concurrency control ensures that transactions
are committed only after passing validation.

The Validation-TS determines the serial execution order,
maintaining database consistency.

"""

    with open(
        filename,
        "w",
        encoding="utf-8"
    ) as file:

        file.write(report)

    print(
        f"Troubleshooting report generated: {filename}"
    )


# ==================================================
# MAIN PROGRAM
# ==================================================

def main():

    print("\n")
    print("*" * 60)

    print(
        "VALIDATION-BASED CONCURRENCY CONTROL"
    )

    print(
        "THREE-PHASE TRANSACTION SIMULATION"
    )

    print("*" * 60)

    # ------------------------------------------------
    # T1: Starts first and reads A, B.
    # ------------------------------------------------

    T1 = Transaction("T1")

    execute_transaction(

        T1,

        read_operations=["A", "B"],

        write_operations={"C": 350}

    )

    # ------------------------------------------------
    # T2: Starts after T1 starts.
    # It reads B and writes B.
    # It will commit before T1 is validated in a
    # concurrent schedule.
    # ------------------------------------------------

    T2 = Transaction("T2")

    execute_transaction(

        T2,

        read_operations=["B"],

        write_operations={"B": 250}

    )

    # ------------------------------------------------
    # T3: Independent transaction.
    # ------------------------------------------------

    T3 = Transaction("T3")

    execute_transaction(

        T3,

        read_operations=["C"],

        write_operations={"D": 450}

    )

    # ------------------------------------------------
    # T1 validation is retried to demonstrate the
    # conflict with T2.
    #
    # T1 started before T2's validation, and its
    # read-set contains B, which T2 wrote.
    # ------------------------------------------------

    print("\n" + "=" * 60)

    print("REVALIDATION DEMONSTRATION FOR T1")

    print("=" * 60)

    # For a clear demonstration, T1's validation is
    # checked after T2 has committed.
    # Its original start timestamp is retained.

    T1.validation_ts = generate_timestamp()

    conflict = (
        T1.read_set &
        set(T2.write_set.keys())
    )

    if conflict:

        T1.status = "ABORTED"

        T1.finish_ts = generate_timestamp()

        failure_details.append(

            "Transaction T1 failed validation.\n"
            "Previous transaction: T2\n"
            f"T1 Read-Set: {sorted(T1.read_set)}\n"
            f"T2 Write-Set: {sorted(T2.write_set.keys())}\n"
            f"Intersection: {sorted(conflict)}\n"
            "Reason: T1 read an item modified by T2.\n"
            "Decision: ABORTED. No T1 writes were committed.\n"

        )

        print(
            "T1 VALIDATION FAILED!"
        )

        print(
            "Conflicting items:",
            conflict
        )

        print(
            "T1 changes were not committed."
        )

        # Update T1's log entry with its final outcome
        for row in transaction_log:

            if row["Transaction_ID"] == "T1":

                row["Validation_TS"] = T1.validation_ts

                row["Finish_TS"] = T1.finish_ts

                row["Status"] = "ABORTED"

    # ------------------------------------------------
    # Generate all deliverables
    # ------------------------------------------------

    generate_csv()

    generate_failure_report()

    # ------------------------------------------------
    # Create ZIP archive
    # ------------------------------------------------

    files = [

        "validation_protocol.py",

        "transaction_log.csv",

        "validation_failure.txt"

    ]

    zip_name = "TASK_2_VALIDATION_PROTOCOL.zip"

    with zipfile.ZipFile(

        zip_name,

        "w",

        zipfile.ZIP_DEFLATED

    ) as archive:

        for filename in files:

            path = Path(filename)

            if path.exists():

                archive.write(
                    path,
                    arcname=filename
                )

    # ------------------------------------------------
    # Final output
    # ------------------------------------------------

    print("\n" + "*" * 60)

    print(
        "FINAL DATABASE STATE"
    )

    print("*" * 60)

    for key, value in database.items():

        print(
            f"{key} = {value}"
        )

    print("\nTRANSACTION EXECUTION ORDER:")

    successful = sorted(

        committed_transactions,

        key=lambda t: t.validation_ts

    )

    for transaction in successful:

        print(

            f"{transaction.transaction_id} "
            f"(Validation-TS: {transaction.validation_ts})"

        )

    print("\nGENERATED FILES:")

    for filename in files:

        print(
            "Created:", filename
        )

    print(
        "\nCreated:", zip_name
    )

    print(
        "\nTASK 2 COMPLETED SUCCESSFULLY!"
    )


if __name__ == "__main__":

    main()



************************************************************
VALIDATION-BASED CONCURRENCY CONTROL
THREE-PHASE TRANSACTION SIMULATION
************************************************************

TRANSACTION: T1
Start-TS: 1

PHASE 1: READ / EXECUTION
T1: Reading A
Read successful: A = 100
T1: Reading B
Read successful: B = 200
T1: Preparing write C = 350

PHASE 2: VALIDATION

Validating T1
Start-TS: 1
Validation-TS: 2
VALIDATION SUCCESSFUL!

PHASE 3: WRITE

Write phase: T1
Database updated: C = 350
Transaction T1 COMMITTED.
Finish-TS: 3

TRANSACTION: T2
Start-TS: 4

PHASE 1: READ / EXECUTION
T2: Reading B
Read successful: B = 200
T2: Preparing write B = 250

PHASE 2: VALIDATION

Validating T2
Start-TS: 4
Validation-TS: 5
VALIDATION SUCCESSFUL!

PHASE 3: WRITE

Write phase: T2
Database updated: B = 250
Transaction T2 COMMITTED.
Finish-TS: 6

TRANSACTION: T3
Start-TS: 7

PHASE 1: READ / EXECUTION
T3: Reading C
Read successful: C = 350
T3: Preparing write D = 450

PHASE 2: VALIDATION

Va